# Lesson 6B: Function Approximation - Practical

## Introduction

In Lesson 6A, we learned that:
- Tabular methods fail in high dimensions (curse of dimensionality)
- Function approximation generalizes across similar states
- The deadly triad (approximation + bootstrapping + off-policy) causes instability

Now we implement and diagnose divergence on **CartPole** and **MountainCar**.

## Setup

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
print('Function Approximation: Practical')

## Tile Coding Implementation

In [ ]:
class TileCoder:
    """Tile coding for discrete state features."""
    
    def __init__(self, state_dims, ranges, num_tilings=8, tile_size=8):
        """
        Args:
            state_dims: number of state dimensions
            ranges: list of (min, max) for each dimension
            num_tilings: number of offset tilings (for generalization)
            tile_size: bins per tiling per dimension
        """
        self.state_dims = state_dims
        self.ranges = ranges
        self.num_tilings = num_tilings
        self.tile_size = tile_size
        self.num_features = num_tilings * (tile_size ** state_dims)
    
    def encode(self, state):
        """Convert continuous state to tile feature vector."""
        features = np.zeros(self.num_features)
        
        for tiling_idx in range(self.num_tilings):
            offset = tiling_idx / self.num_tilings
            tile_idx = 0
            
            for dim, (s, (min_val, max_val)) in enumerate(zip(state, self.ranges)):
                # Normalize to [0, 1]
                norm_s = (s - min_val) / (max_val - min_val + 1e-8)
                # Apply offset for this tiling
                offset_s = (norm_s + offset) % 1.0
                # Convert to tile bin
                bin_idx = int(offset_s * self.tile_size)
                bin_idx = min(max(bin_idx, 0), self.tile_size - 1)
                # Update multi-dimensional index
                tile_idx = tile_idx * self.tile_size + bin_idx
            
            # Set feature bit for this tiling
            feature_idx = tiling_idx * (self.tile_size ** self.state_dims) + tile_idx
            if feature_idx < self.num_features:
                features[feature_idx] = 1.0
        
        return features

# Test on CartPole
env = gym.make('CartPole-v1')
print(f'CartPole observation space: {env.observation_space}')
print(f'CartPole action space: {env.action_space}')

coder = TileCoder(
    state_dims=4,
    ranges=[(-2.4, 2.4), (-3.0, 3.0), (-0.2, 0.2), (-2.0, 2.0)],
    num_tilings=8,
    tile_size=8
)
print(f'Tile coder: {coder.num_features} features')

# Test encoding
state, _ = env.reset()
features = coder.encode(state)
print(f'Sample features shape: {features.shape}, non-zero: {np.count_nonzero(features)}')

## Sarsa with Tile Coding (On-Policy)

On-policy is stable; off-policy (Q-learning) can diverge with function approximation.

In [ ]:
class SarsaTileCoding:
    """Sarsa with tile coding on CartPole."""
    
    def __init__(self, coder, num_actions, alpha=0.01, gamma=0.99, epsilon=0.1):
        self.coder = coder
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.w = np.zeros((num_actions, coder.num_features))
    
    def get_action(self, state):
        """ε-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            features = self.coder.encode(state)
            return np.argmax([np.dot(self.w[a], features) for a in range(self.num_actions)])
    
    def get_value(self, state, action):
        """Value estimate for state-action pair."""
        features = self.coder.encode(state)
        return np.dot(self.w[action], features)
    
    def update(self, state, action, reward, next_state, next_action, done):
        """Sarsa update step."""
        features = self.coder.encode(state)
        features_next = self.coder.encode(next_state)
        
        v = np.dot(self.w[action], features)
        v_next = np.dot(self.w[next_action], features_next) if not done else 0.0
        delta = reward + self.gamma * v_next - v
        
        # Update weights for this action
        self.w[action] += self.alpha * delta * features

# Train Sarsa
print('Training Sarsa with tile coding on CartPole...')
agent = SarsaTileCoding(coder, num_actions=2, alpha=0.01, gamma=0.99)
returns_sarsa = []

for ep in range(100):
    state, _ = env.reset()
    action = agent.get_action(state)
    episode_return = 0
    
    for step in range(500):
        next_state, reward, done, _, _ = env.step(action)
        next_action = agent.get_action(next_state)
        
        agent.update(state, action, reward, next_state, next_action, done)
        
        episode_return += reward
        state, action = next_state, next_action
        
        if done:
            break
    
    returns_sarsa.append(episode_return)

print(f'Sarsa final return: {returns_sarsa[-1]:.1f}')

## Neural Network Approximator

Now try a neural network value function on the same CartPole.

In [ ]:
class SimpleNN:
    """Simple neural network value function (2 hidden layers)."""
    
    def __init__(self, input_size, hidden_size=64, learning_rate=0.001):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        
        # Simple linear layers (no true backprop, just for illustration)
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b2 = np.zeros(hidden_size)
        self.W3 = np.random.randn(hidden_size, 1) * 0.01
        self.b3 = np.zeros(1)
    
    def forward(self, x):
        """Forward pass through network."""
        self.h1 = np.maximum(0, np.dot(x, self.W1) + self.b1)  # ReLU
        self.h2 = np.maximum(0, np.dot(self.h1, self.W2) + self.b2)  # ReLU
        out = np.dot(self.h2, self.W3) + self.b3
        return out[0]
    
    def backward_approx(self, x, delta):
        """Approximate gradient descent update (semi-gradient)."""
        # Simplified: just update final layer with approximate gradient
        h1 = np.maximum(0, np.dot(x, self.W1) + self.b1)
        h2 = np.maximum(0, np.dot(h1, self.W2) + self.b2)
        
        # Update output layer
        grad_W3 = np.outer(h2, [delta])
        self.W3 += self.lr * grad_W3
        self.b3 += self.lr * delta

# Train neural network with Q-learning (will likely diverge)
print('Training neural network with Q-learning on CartPole...')
nn_agent = SimpleNN(input_size=4, hidden_size=32, learning_rate=0.01)
returns_nn = []
q_values = []

for ep in range(100):
    state, _ = env.reset()
    episode_return = 0
    ep_q_values = []
    
    for step in range(500):
        # ε-greedy
        if np.random.rand() < 0.1:
            action = np.random.randint(2)
        else:
            v0 = nn_agent.forward(state)
            # Estimate Q for action 1 (simplified)
            state_copy = state.copy()
            state_copy[0] += 0.01  # Perturb slightly
            v1 = nn_agent.forward(state_copy)
            action = 1 if v1 > v0 else 0
        
        next_state, reward, done, _, _ = env.step(action)
        
        # Q-learning update (can diverge)
        q_current = nn_agent.forward(state)
        q_next = nn_agent.forward(next_state) if not done else 0.0
        delta = reward + 0.99 * q_next - q_current
        
        nn_agent.backward_approx(state, delta)
        
        ep_q_values.append(abs(q_current))
        episode_return += reward
        state = next_state
        
        if done:
            break
    
    returns_nn.append(episode_return)
    q_values.append(np.mean(ep_q_values) if ep_q_values else 0)

print(f'NN Q-learning final return: {returns_nn[-1]:.1f}')
print(f'Average Q-value trend: {np.mean(q_values[-10:]):.2f} (diverging if growing unbounded)')

## Comparison: Tile Coding vs Neural Network

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves
ax = axes[0]
window = 10
sarsa_smooth = np.convolve(returns_sarsa, np.ones(window)/window, mode='valid')
nn_smooth = np.convolve(returns_nn, np.ones(window)/window, mode='valid')
ax.plot(sarsa_smooth, label='Sarsa + Tile Coding (On-Policy)', linewidth=2)
ax.plot(nn_smooth, label='NN + Q-Learning (Off-Policy)', linewidth=2, alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.set_title('Function Approximation Comparison on CartPole')
ax.legend()
ax.grid(True, alpha=0.3)

# Q-value divergence trend
ax = axes[1]
ax.plot(q_values, linewidth=2, color='red', alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('Mean |Q-Value|')
ax.set_title('Neural Network Q-Values: Divergence Signal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nKey Observation:')
print(f'  Sarsa (on-policy) final return: {returns_sarsa[-1]:.1f}')
print(f'  NN Q-learning (off-policy) final return: {returns_nn[-1]:.1f}')
print(f'  Deadly triad: approximation + bootstrapping + off-policy → divergence')

## Stable-Baselines3 Reference: PPO (with Value Function Approximation)

Stable-Baselines3 uses advanced techniques to stabilize learning with neural networks.

In [ ]:
from stable_baselines3 import PPO

# Train PPO on CartPole
print('Training Stable-Baselines3 PPO on CartPole...')
model = PPO('MlpPolicy', env, learning_rate=3e-4, n_steps=2048, verbose=0)
model.learn(total_timesteps=50000)

# Evaluate
mean_reward = 0
num_evals = 20
for _ in range(num_evals):
    obs, _ = env.reset()
    episode_reward = 0
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _, _ = env.step(action)
        episode_reward += reward
    mean_reward += episode_reward

mean_reward /= num_evals
print(f'PPO mean evaluation return: {mean_reward:.1f}')
print(f'\nWhy PPO succeeds where naive Q-learning failed:')
print('  1. On-policy: uses recent data, avoids deadly triad divergence')
print('  2. Clipped surrogate: bounds policy update, adds stability')
print('  3. Multiple epochs: efficient use of samples')
print('  4. GAE: generalized advantage estimation (eligibility traces concept!)')

## Summary

### What We Learned

1. **Tile Coding**: Simple, interpretable, hand-crafted features → stable on-policy learning
2. **Neural Networks**: End-to-end learning, but naive Q-learning diverges (deadly triad)
3. **Stabilization Tricks**:
   - Experience replay (breaks temporal correlation)
   - Target network (non-stationary target problem)
   - Clipping (PPO, TRPO)
   - GAE (elegibility traces in modern form)

### The Bridge

- Lesson 5: Eligibility traces (efficient credit assignment)
- Lesson 6: Function approximation (scalability)
- Lesson 7 (DQN): Combines both with experience replay + target networks
- Lesson 8 (Policy Gradients): PPO uses gradient descent directly on policy

### Key Insight

Function approximation + bootstrapping + off-policy = deadly triad → need stabilization. Different algorithms use different fixes:
- DQN: Experience replay + target network + ε-decay
- PPO: On-policy (avoids deadly triad) + clipping
- A3C: On-policy + multiple parallel workers

Modern deep RL all rely on these foundations.